# Exercise 1: image → JSON → DataFrame

## Tasks

1. **One image:** send a single picture to the API and read its description.
2. **Structure:** ask for a fixed JSON format with a few fields.
3. **More images:** repeat for a small folder and collect the answers in a table.
4. **Evaluate:** compare the table with hand labels, field by field.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
# Standard Python libraries.
import base64
import json
import mimetypes
import os
from pathlib import Path
from typing import Literal

# Libraries installed from requirements.txt.
import jiwer
import pandas as pd
from openai import OpenAI
from PIL import Image
from pydantic import BaseModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("01_Image_Extraction")

# Connect to OpenAI. The key is read from OPENAI_API_KEY; busy requests are retried.
client = OpenAI(max_retries=5, timeout=60)
MODEL = "gpt-6-luna"

# Folder with the course images and videos.
DATA = Path("..") / "data"

## 1. Send one image to the API and read its description

In [ ]:
# Find all course images.
folder = DATA / "images"
paths = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
print(len(paths), "images")

# Show the first image at a quarter of its size.
display(Image.open(paths[0]).reduce(4))

The API request is text, so the image bytes are converted to text (base64). This function does it for any image file:

In [ ]:
# Function 1: turn an image file into an API input.
def image_to_input(path):
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }

In [ ]:
# Convert the first image and look at the start of the text.
image_input = image_to_input(paths[0])
print(image_input["image_url"][:80], "...")

In [ ]:
# Combine a question and the image in one message.
QUESTION = "Describe this image in two sentences."
message = {
    "role": "user",
    "content": [{"type": "input_text", "text": QUESTION}, image_input],
}

# Send the message and print the answer.
response = client.responses.create(
    model=MODEL,
    input=[message],
    reasoning={"effort": "none"},  # no extra thinking: faster and cheaper
    store=False,  # OpenAI does not keep the answer
)
print(response.output_text)

The image is read as tokens, so every image costs money. Estimate before you run: cost per image × number of images.

In [ ]:
# Prices for this model.
PRICE_INPUT = 0.10   # US$ per 1 million input tokens
PRICE_OUTPUT = 0.50  # US$ per 1 million output tokens

# Tokens used by this call, the cost, and the cost for all images.
usage = response.usage
cost = (usage.input_tokens * PRICE_INPUT + usage.output_tokens * PRICE_OUTPUT) / 1_000_000
print(usage.input_tokens, "input tokens,", usage.output_tokens, "output tokens")
print(f"This image: US$ {cost:.5f}")
print(f"All {len(paths)} images: about US$ {cost * len(paths):.4f}")

## 2. Ask for a fixed JSON format

A free text is hard to compare. Now the answer must contain exactly these fields, written as a Python class. `None` means no text is visible.

In [ ]:
# The answer format: one field per piece of information.
class ImageRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    objects: list[str]
    people_count: int
    text_in_image: str | None


# The instruction for the model.
PROMPT = (
    "Describe this image. Pick the scene from the list. "
    "Name up to five prominent objects as lowercase singular nouns. "
    "Count the visible people. "
    "Copy the most prominent text. Use null if there is no text "
    "and [unreadable] if the text cannot be read."
)

In [ ]:
# Same image, new prompt.
message = {
    "role": "user",
    "content": [{"type": "input_text", "text": PROMPT}, image_input],
}

# parse() checks the answer against ImageRecord.
response = client.responses.parse(
    model=MODEL,
    input=[message],
    text_format=ImageRecord,
    reasoning={"effort": "none"},
    store=False,
)
print(response.output_text)  # the raw answer: JSON text

# Get the parsed answer and stop if there is none.
answer = response.output_parsed
if answer is None:
    raise ValueError("No valid answer. Look at the raw answer above.")
print(answer.scene, "|", answer.objects)

## 3. Repeat for all images and collect the answers in a table

The steps from above, put into one function. It also writes every answer to `api_log.jsonl`.

In [ ]:
# Function 2: send a prompt and images, get the answer as a dict.
def ask(prompt, answer_format, images):
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Stop with a clear message if there is no valid answer.
    if response.output_parsed is None:
        raise ValueError(f"No valid answer for {images}. Look at api_log.jsonl.")

    # 5. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

In [ ]:
# Ask the model about every image and add the filename ourselves.
rows = []
for path in paths:
    row = ask(PROMPT, ImageRecord, [path])
    row["file"] = path.name
    rows.append(row)

# One row per image; save the table as JSON lines.
extracted = pd.DataFrame(rows)
extracted.to_json("extracted.jsonl", orient="records", lines=True)
extracted

## 4. Compare the table with the hand labels, field by field

The average hides the weak field – so each field gets its own score.

In [ ]:
# Hand labels: one row per image. Objects are separated by ";".
labels = pd.read_csv(folder / "labels.csv", keep_default_na=False)

# Put prediction and label side by side, matched by filename.
scores = extracted.merge(labels, on="file", suffixes=("_pred", "_ref"))

# Exact match: share of images where model and label agree.
print("Scene accuracy:       ", accuracy_score(scores["scene_ref"], scores["scene_pred"]))
print("People count accuracy:", accuracy_score(scores["people_count_ref"], scores["people_count_pred"]))

**Objects** are a list, so an exact match is too strict. F1 combines
- *precision*: how many of the named objects are correct?
- *recall*: how many of the true objects were found?

In [ ]:
# Object lists from the model and from the labels.
pred = scores["objects_pred"]
ref = scores["objects_ref"].str.split(";")  # "boat;water" -> ["boat", "water"]

# Turn each list into a row of 0s and 1s, one column per object name.
encoder = MultiLabelBinarizer()
encoder.fit(pd.concat([pred, ref]))
y_pred = encoder.transform(pred)
y_ref = encoder.transform(ref)

# Averages over all images.
print("Object precision:", round(precision_score(y_ref, y_pred, average="samples", zero_division=0), 2))
print("Object recall:   ", round(recall_score(y_ref, y_pred, average="samples", zero_division=0), 2))
print("Object F1:       ", round(f1_score(y_ref, y_pred, average="samples", zero_division=0), 2))

**Text** is scored with the character error rate (CER): how many characters must be changed, per character of the label? Lower is better.

In [ ]:
# Keep only images whose label contains readable text.
with_text = scores[~scores["text_in_image_ref"].isin(["null", "[unreadable]"])]

# Compare model text and label text character by character.
reference = list(with_text["text_in_image_ref"])
prediction = list(with_text["text_in_image_pred"].fillna(""))  # no answer = empty text
print("Model:", prediction, "| Label:", reference)
print("CER:", round(jiwer.cer(reference, prediction), 2))

**Check yourself:** Why can a perfectly valid JSON answer still be wrong? Compare the two object lists of a few images.

In [ ]:
# Model and label objects side by side.
scores[["file", "objects_pred", "objects_ref"]]

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')